# Speech Separation Pipeline - Kaggle Runner

**Before running this notebook:**
1. Accept gated model terms on BOTH of these pages (logged in with the same HF account as your token):
   - https://huggingface.co/pyannote/speaker-diarization-3.1
   - https://huggingface.co/pyannote/segmentation-3.0
2. Create a fresh HF token at https://huggingface.co/settings/tokens (read access is enough).
3. In this notebook: left sidebar → Add-ons → Secrets → add a secret named `ADD_TOKEN_NAME` with that token value.
4. Set Accelerator to **GPU T4 x2**
5. Make sure your `speech_separation` files are attached as a Kaggle Input dataset (folder containing main.py, cascade.py, config.py, diarization.py, separator.py, postprocess.py, requirements.txt, test_pipeline.py).

In [ ]:
# STEP 1: Load HF token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN_2')
print("HF_TOKEN loaded:", bool(os.environ.get('HF_TOKEN')))

In [ ]:
# STEP 2: Locate your uploaded project files automatically
import subprocess

result = subprocess.run(['find', '/kaggle/input', '-name', 'main.py'], capture_output=True, text=True)
found_paths = result.stdout.strip().split('\n')
print("Found main.py at:")
for p in found_paths:
    print(" ", p)

assert found_paths and found_paths[0], "Could not find main.py under /kaggle/input — check your dataset is attached."
SRC_DIR = found_paths[0].rsplit('/', 1)[0]
print("\nUsing source directory:", SRC_DIR)

In [ ]:
# STEP 3: Copy project into working directory
import shutil, os

DEST_DIR = '/kaggle/working/speech_separation'
if os.path.exists(DEST_DIR):
    shutil.rmtree(DEST_DIR)
shutil.copytree(SRC_DIR, DEST_DIR)

os.chdir(DEST_DIR)
print("Now in:", os.getcwd())
print(os.listdir('.'))

In [ ]:
# STEP 3b: Pin numpy/scipy in requirements.txt before the one-and-only install
# (avoids a second live reinstall after imports have already happened, which is what
# corrupted the kernel last time). These versions satisfy pyannote-metrics/pyannote-core's
# numpy>=2.2.2 floor and scipy's own numpy<2.5 ceiling for scipy 1.15.x.
req_path = os.path.join(DEST_DIR, 'requirements.txt')
with open(req_path) as f:
    lines = [l for l in f if not l.strip().lower().startswith(('numpy', 'scipy'))]
lines += ['numpy==2.2.6\n', 'scipy==1.15.2\n']
with open(req_path, 'w') as f:
    f.writelines(lines)
print("requirements.txt now pins:")
!grep -in "numpy\|scipy" {req_path}

In [ ]:
# STEP 4: Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# STEP 5: Defensively patch diarization.py for pyannote 4.x API

import re

diar_path = os.path.join(DEST_DIR, 'diarization.py')
with open(diar_path, 'r') as f:
    content = f.read()

original = content

# --- Patch 1: use_auth_token -> token ---
content = content.replace('use_auth_token', 'token')

# --- Patch 2: insert unwrap helper if not already present ---
helper = (
    "def _unwrap_diarization(diarization):\n"
    "    \"\"\"pyannote.audio 4.x returns DiarizeOutput; 3.x returns Annotation directly.\"\"\"\n"
    "    return diarization.speaker_diarization if hasattr(diarization, 'speaker_diarization') else diarization\n\n"
)
if '_unwrap_diarization' not in content:
    # insert helper right after the imports block (after last top-level import line)
    lines = content.split('\n')
    insert_at = 0
    for i, line in enumerate(lines):
        if line.startswith('import ') or line.startswith('from '):
            insert_at = i + 1
    lines.insert(insert_at, '\n' + helper)
    content = '\n'.join(lines)

# --- Patch 3: rewrite every "<var>.itertracks(" call to unwrap first ---
def rewrite_itertracks(match):
    varname = match.group(1)
    return f'_unwrap_diarization({varname}).itertracks('

content = re.sub(r'(\b\w+)\.itertracks\(', rewrite_itertracks, content)
# avoid double-wrapping if this cell runs twice
content = content.replace('_unwrap_diarization(_unwrap_diarization(', '_unwrap_diarization((')

if content != original:
    with open(diar_path, 'w') as f:
        f.write(content)
    print("Patched diarization.py: token= rename + itertracks unwrap")
else:
    print("No changes needed (already patched).")

print("\n--- token/auth lines ---")
!grep -n "token" {diar_path}
print("\n--- itertracks lines ---")
!grep -n "itertracks" {diar_path}
print("\n--- helper ---")
!grep -n "_unwrap_diarization" {diar_path}

In [ ]:
# STEP 5b: Patch postprocess.py — fix CUDA/CPU device mismatch in extract_embedding()
# resampler was instantiated on CPU but waveform tensors are on CUDA during dedup.
# Fix: move the resampler to the waveform's device at call time (cheap, resampler is tiny).

postprocess_path = os.path.join(DEST_DIR, 'postprocess.py')
with open(postprocess_path, 'r') as f:
    content = f.read()

original = content
content = content.replace(
    'wav_16k = resampler(waveform)',
    'wav_16k = resampler.to(waveform.device)(waveform)'
)

if content != original:
    with open(postprocess_path, 'w') as f:
        f.write(content)
    print("Patched postprocess.py: resampler moved to waveform.device before call")
else:
    print("Pattern not found — check exact line, may need manual look")

!grep -n "resampler" {postprocess_path}

In [ ]:
# STEP 5c: Patch main.py — torchaudio.save() requires a CPU tensor,
# but norm_wav is still on CUDA at save time (torchcodec backend doesn't support CUDA input).

main_path = os.path.join(DEST_DIR, 'main.py')
with open(main_path, 'r') as f:
    content = f.read()

original = content
content = content.replace(
    'torchaudio.save(out_path, norm_wav, sample_rate)',
    'torchaudio.save(out_path, norm_wav.detach().cpu(), sample_rate)'
)

if content != original:
    with open(main_path, 'w') as f:
        f.write(content)
    print("Patched main.py: norm_wav moved to CPU before torchaudio.save()")
else:
    print("Pattern not found — check exact line, may need manual look")

!grep -n "torchaudio.save" {main_path}

In [ ]:
# STEP 5d: Patch diarization.py — require minimum cumulative duration per speaker
# label before counting it, instead of counting any label pyannote emits even once.
# This targets over-recursion caused by SepFormer leakage artifacts getting
# mislabeled as a spurious "second speaker" by pyannote.

diar_path = os.path.join(DEST_DIR, 'diarization.py')
with open(diar_path, 'r') as f:
    content = f.read()

original = content
old_block = '''        # extract unique speaker labels
        speakers = set()
        for turn, _, speaker in _unwrap_diarization(diarization).itertracks(yield_label=True):
            speakers.add(speaker)
            
        return len(speakers)'''

new_block = '''        # accumulate total speaking duration per label — a label that only ever
        # appears in a tiny/spurious segment (e.g. separation leakage artifact)
        # should not count as a real second speaker
        speaker_durations = {}
        for turn, _, speaker in _unwrap_diarization(diarization).itertracks(yield_label=True):
            speaker_durations[speaker] = speaker_durations.get(speaker, 0.0) + (turn.end - turn.start)

        real_speakers = {
            spk for spk, dur in speaker_durations.items()
            if dur >= config.MIN_SPEAKER_DURATION_SEC
        }
        return len(real_speakers)'''

content = content.replace(old_block, new_block)

if content != original:
    with open(diar_path, 'w') as f:
        f.write(content)
    print("Patched diarization.py: min-duration-per-speaker filter added")
else:
    print("Pattern not found — paste current file contents, block may not match exactly")

!grep -n "MIN_SPEAKER_DURATION_SEC\|real_speakers" {diar_path}

In [ ]:
# STEP 5e: Add MIN_SPEAKER_DURATION_SEC to config.py
config_path = os.path.join(DEST_DIR, 'config.py')
with open(config_path, 'r') as f:
    content = f.read()

if 'MIN_SPEAKER_DURATION_SEC' not in content:
    content += '\nMIN_SPEAKER_DURATION_SEC = 0.5  # seconds; ignore speaker labels with less total speech than this\n'
    with open(config_path, 'w') as f:
        f.write(content)
    print("Added MIN_SPEAKER_DURATION_SEC = 0.5 to config.py")
else:
    print("Already present")

!grep -n "MIN_SPEAKER_DURATION_SEC" {config_path}

In [ ]:
# STEP 5f: Patch postprocess.py — add cap_max_channels(), keeps the N loudest
# channels (by RMS) after dedup. Same energy proxy already used for merge-keep logic.

postprocess_path = os.path.join(DEST_DIR, 'postprocess.py')
with open(postprocess_path, 'r') as f:
    content = f.read()

original = content
anchor = "    def normalize_loudness(self, waveform):"
new_method = '''    def cap_max_channels(self, channels_dict, max_channels=None):
        """
        If more than max_channels survive pruning+dedup, keep only the
        max_channels loudest (by RMS) — a proxy for "clearest/strongest",
        since we have no transcription/intelligibility signal available.
        channels_dict: { 'id': waveform_tensor }
        Returns: capped_dict, dropped_ids
        """
        if max_channels is None:
            max_channels = config.MAX_OUTPUT_CHANNELS

        if len(channels_dict) <= max_channels:
            return channels_dict, []

        energies = {cid: self._compute_rms_db(wav) for cid, wav in channels_dict.items()}
        ranked = sorted(channels_dict.keys(), key=lambda c: energies[c], reverse=True)
        keep_ids = set(ranked[:max_channels])
        dropped = [(cid, energies[cid]) for cid in ranked[max_channels:]]

        capped_dict = {cid: channels_dict[cid] for cid in channels_dict if cid in keep_ids}
        return capped_dict, dropped

'''

if 'cap_max_channels' not in content:
    content = content.replace(anchor, new_method + anchor)
    with open(postprocess_path, 'w') as f:
        f.write(content)
    print("Patched postprocess.py: cap_max_channels() added")
else:
    print("Already present")

!grep -n "cap_max_channels\|def normalize_loudness" {postprocess_path}

In [ ]:
# STEP 5g: Add MAX_OUTPUT_CHANNELS to config.py

config_path = os.path.join(DEST_DIR, 'config.py')
with open(config_path, 'r') as f:
    content = f.read()

if 'MAX_OUTPUT_CHANNELS' not in content:
    content += '\nMAX_OUTPUT_CHANNELS = 5  # spec ceiling: never output more than this many speakers\n'
    with open(config_path, 'w') as f:
        f.write(content)
    print("Added MAX_OUTPUT_CHANNELS = 5 to config.py")
else:
    print("Already present")

!grep -n "MAX_OUTPUT_CHANNELS" {config_path}

In [ ]:
# STEP 5h: Patch main.py — call cap_max_channels() after dedup, before saving

main_path = os.path.join(DEST_DIR, 'main.py')
with open(main_path, 'r') as f:
    content = f.read()

original = content
old_block = '''    print(f"  Final channel count after pruning and dedup: {len(dedup_dict)}")
    
    # 8. Output Normalization and Saving'''

new_block = '''    print(f"  Final channel count after pruning and dedup: {len(dedup_dict)}")

    # 7b. Cap output at MAX_OUTPUT_CHANNELS, keeping the loudest/strongest
    dedup_dict, capped_dropped = postprocessor.cap_max_channels(dedup_dict)
    if capped_dropped:
        print(f"\\n[INFO] Capping to {config.MAX_OUTPUT_CHANNELS} strongest channels:")
        for cid, rms in capped_dropped:
            print(f"  - Dropped {cid} for output cap ({rms:.2f} dBFS, weaker signal)")
    
    # 8. Output Normalization and Saving'''

content = content.replace(old_block, new_block)

if content != original:
    with open(main_path, 'w') as f:
        f.write(content)
    print("Patched main.py: cap_max_channels() call inserted after dedup")
else:
    print("Pattern not found — block may not match exactly, paste current main.py")

!grep -n "cap_max_channels\|MAX_OUTPUT_CHANNELS" {main_path}

In [ ]:
# STEP 5i: Patch main.py — clear outdir before writing, so stale files from a

main_path = os.path.join(DEST_DIR, 'main.py')
with open(main_path, 'r') as f:
    content = f.read()

original = content
old_block = '''    if not os.path.exists(outdir):
        os.makedirs(outdir)'''

new_block = '''    if os.path.exists(outdir):
        import shutil as _shutil
        _shutil.rmtree(outdir)
    os.makedirs(outdir)'''

content = content.replace(old_block, new_block)

if content != original:
    with open(main_path, 'w') as f:
        f.write(content)
    print("Patched main.py: outdir cleared before each run")
else:
    print("Pattern not found — paste current main.py top section")

!grep -n "rmtree\|makedirs" {main_path}

In [ ]:
# STEP 6: Sanity check GPU is visible
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

In [ ]:
# STEP 7: Run the smoke test (dummy sine-wave audio, just checks pipeline runs end-to-end)
#!python test_pipeline.py

## STEP 8: Run on a real mixture file

Upload a real 3-5 speaker overlapping `.wav` file as a Kaggle Input dataset (or add it to the same dataset as your code), then run the cell below with the correct path.

In [ ]:
!find /kaggle/input -iname "*.wav"

In [ ]:
# STEP 8: Run on real audio — EDIT the input path to point at your real mixture file
# To the AIMS sr. :
# Attach your own test audio as a Kaggle Input dataset, then run the find cell above to get the real path, and paste it here."
REAL_INPUT = "/kaggle/input/datasets/bluedeath/3speaker2/sample_2.wav"  # <-- EDIT THIS!!!!
OUTDIR = "/kaggle/working/outputs"

!python main.py --input "{REAL_INPUT}" --outdir "{OUTDIR}" 

In [ ]:
# STEP 8b: DIAGNOSTIC — print full pairwise similarity matrix for the last run's candidates

import sys
sys.path.insert(0, DEST_DIR)
import postprocess, torch, torchaudio, os

pp = postprocess.PostProcessor()  
OUTDIR = "/kaggle/working/outputs"
wavs = {}
for fname in sorted(os.listdir(OUTDIR)):
    if fname.endswith('.wav'):
        wav, sr = torchaudio.load(os.path.join(OUTDIR, fname))
        wavs[fname] = wav

embeddings = {name: pp.extract_embedding(wav) for name, wav in wavs.items()}
names = list(embeddings.keys())
print(f"{'':20s}" + "".join(f"{n[:12]:>14s}" for n in names))
for a in names:
    row = f"{a:20s}"
    for b in names:
        sim = torch.nn.functional.cosine_similarity(embeddings[a].flatten(), embeddings[b].flatten(), dim=0).item()
        row += f"{sim:14.3f}"
    print(row)

In [ ]:
# STEP 9: Listen to the separated outputs
import IPython.display as ipd
import os

OUTDIR = "/kaggle/working/outputs"
if os.path.exists(OUTDIR):
    for fname in sorted(os.listdir(OUTDIR)):
        if fname.endswith('.wav'):
            print(fname)
            display(ipd.Audio(os.path.join(OUTDIR, fname)))
else:
    print("Output directory not found yet — run Step 8 first.")